# 08 · Business Interpretation, Recommendations and Reproducibility
### IT3091 Machine Learning · Group 2026-AI-42

Consolidated results, operational recommendations, risk mitigations, decision governance, and environment reproducibility parameters.


In [1]:
import sys, pathlib, warnings
ROOT = pathlib.Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import json, platform
import numpy as np, pandas as pd
from IPython.display import display
import sklearn

from src.config import *
from src.data.loading import fingerprint
from src.data.prepare import load_modelling_data

pd.set_option("display.max_columns", 60, "display.width", 170, "display.max_colwidth", 140)
data = load_modelling_data()
y_test, y_train = data["y_test"], data["y_train"]
T = lambda name, **kw: pd.read_csv(TABLES_DIR / name, **kw)
params = json.loads((ARTIFACTS_DIR / "tuned_params.json").read_text())
labels = json.loads((ARTIFACTS_DIR / "policy_labels.json").read_text())
TUNED_NAME = params["name"]

Loading local file: bank-additional-full.csv


Rows: raw 41,188 -> cleaned 41,176 | train 32,940 (rate 6.38%) | pilot 500 | test 7,736 (rate 32.41%)


---
## 1 · Summary of Key Results


In [2]:
holdout = T("holdout_results.csv", index_col=0)
policies = T("policy_comparison.csv", index_col=0)
blocks = T("test_blocks.csv", index_col=0)
leak = T("preprocessing_leakage_duration.csv", index_col=0)
split_opt = T("split_optimism.csv", index_col=0)

best_row = holdout.loc[TUNED_NAME]
top, rnd, allc = policies.loc[labels["P_BUDGET"]], policies.loc[labels["P_RANDOM"]], policies.loc[labels["P_ALL"]]
print("=== Key numbers for the report ===")
print(f"FINAL MODEL: {TUNED_NAME} (selected on forward-chaining PR-AUC in the training period, notebooks 03-04)")
print(f"CLIENT-LEVEL SKILL (within time blocks): ROC-AUC {blocks['auc'].mean():.3f} | lift@{CALL_BUDGET:.0%} {blocks['lift_at_budget'].mean():.2f}x")
print(f"Capacity policy (top {CALL_BUDGET:.0%} of each block): captures {top.subscribers_captured_pct:.1f}% of subscribers "
      f"(precision {top.precision:.3f} vs {y_test.mean():.3f} at random = {top.precision / y_test.mean():.2f}x)")
print(f"Profit per client at a {CALL_BUDGET:.0%} call capacity: model {top.profit_per_client:.3f} vs random {rnd.profit_per_client:.3f} | call everyone {allc.profit_per_client:.3f}")
print(f"GLOBAL ranking over the whole test period (includes the period effect; do not headline): ROC-AUC {best_row.roc_auc:.3f} | PR-AUC {best_row.pr_auc:.3f} | lift@{CALL_BUDGET:.0%} {best_row.lift_at_budget:.2f}x")
print(f"Leakage check: ROC-AUC with duration {leak.iloc[0].roc_auc:.3f} vs without {leak.iloc[1].roc_auc:.3f}")
opt = split_opt.iloc[:3][["roc_auc", "lift_at_budget"]].mean() - split_opt.iloc[3][["roc_auc", "lift_at_budget"]]
print(f"Optimism of a stratified random split vs the chronological hold-out: ROC-AUC {opt['roc_auc']:+.3f}, lift {opt['lift_at_budget']:+.2f}x")

=== Key numbers for the report ===
FINAL MODEL: Tuned Logistic regression (selected on forward-chaining PR-AUC in the training period, notebooks 03-04)
CLIENT-LEVEL SKILL (within time blocks): ROC-AUC 0.609 | lift@20% 1.35x
Capacity policy (top 20% of each block): captures 26.9% of subscribers (precision 0.436 vs 0.324 at random = 1.35x)
Profit per client at a 20% call capacity: model 0.673 vs random 0.448 | call everyone 2.241
GLOBAL ranking over the whole test period (includes the period effect; do not headline): ROC-AUC 0.689 | PR-AUC 0.499 | lift@20% 1.72x
Leakage check: ROC-AUC with duration 0.953 vs without 0.671
Optimism of a stratified random split vs the chronological hold-out: ROC-AUC +0.073, lift +1.09x


---
## 2 · Strategic Recommendations

1. **Deploy Rank-Based Capacity Targeting**: Target the top 20% scored clients within each operational call window to maximize subscriber conversion within call center capacity.
2. **Strict Exclusion of Post-Call Information**: Ensure `duration` remains excluded from all operational models to prevent catastrophic data leakage.
3. **Segment-Informed Scripting**: Utilize identified demographic clusters to customize sales conversations while maintaining model-driven call ordering.
4. **Dynamic Drift Monitoring**: Track recent batch conversion rates and apply rolling-intercept adjustments to recalibrate probability estimates continuously.
5. **Fatigue Management**: Restrict repeated contacts per individual to prevent customer dissatisfaction and diminishing returns.
6. **Ethical Governance**: Review fairness metrics quarterly to prevent unintentional bias against demographic sub-groups.

### Limitations and Risks
- **Macro Drift**: Economic conditions shift over time; models require periodic retraining on recent campaign data.
- **Assumed Economics**: Profit estimations rely on illustrative cost-value ratios and should be mapped to actual operational cost structures.
- **Customer Identifiers**: Lack of persistent customer IDs limits longitudinal multi-campaign tracking.
- **Correlation vs Causality**: Scores predict natural propensity to respond, not uplift strictly caused by contact.


---
## 3 · Project Decision Log

| # | Decision Area | Options Considered | Final Choice | Rationale & Evidence |
|---|---|---|---|---|
| D1 | Primary Formulation | Response prediction / Segmentation / Profile analysis | Campaign response prediction | Supervised binary classification aligns directly with call center prioritization |
| D2 | Secondary Formulation | None / Segmentation | Segment prioritization | Enhances operational scripting without replacing granular scoring (Notebook 07) |
| D3 | Feature Leakage (`duration`) | Retain / Exclude | Exclude | Post-call metric creates artificial skill; dropping it ensures deployability (Notebook 02) |
| D4 | Missing Values (`unknown`) | Drop / Impute / Dedicated Category | Dedicated category | Preserves sample volume and captures missingness signal (Notebook 02) |
| D5 | `pdays = 999` Handling | Continuous / Null / Binary Flag | Indicator flag + zero | Accurately models clients never previously contacted (Notebook 02) |
| D6 | Metric Selection | Accuracy / ROC-AUC / PR-AUC / Lift / Profit | PR-AUC, ROC-AUC, Lift@20%, Profit | Robust to class imbalance and directly maps to campaign economics (Notebook 01, 06) |
| D7 | Validation Strategy | Random K-Fold / Chronological Split | Chronological split + forward chaining | Reflects real-world temporal deployment and avoids split optimism (Notebook 01, 05) |
| D8 | Class Imbalance | None / Class Weights / SMOTE / SMOTE-Tomek | Class weighting / Simplest balanced | Evaluated on forward-chaining folds; preserves calibration (Notebook 04) |
| D9 | Selected Architecture | LR / RF / HGB / MLP / Challengers | Regularized Logistic Regression / Tuned Pipeline | Highest consistent forward-chaining PR-AUC and transparent coefficients (Notebook 03-05) |
| D10 | Probability Calibration | None / Isotonic / Platt / Rolling Intercept | Rolling intercept recalibration | Adapts to drifting base rates without altering ranking integrity (Notebook 06) |
| D11 | Macroeconomic Indicators | Include / Exclude | Exclude by default | Acts as temporal proxy; models without macro show greater transferability (Notebook 03) |
| D12 | Engineered Features | Baseline / Individual / All | Selective feature inclusion | Retained only candidate feature families demonstrating CV improvement (Notebook 02) |
| D13 | Model Exploration | Incumbent vs Challengers | Incumbent retained | Challengers did not surpass tuned baseline across all evaluation folds (Notebook 03, 05) |
| D14 | Operational Policy | Score threshold vs Capacity ranking | Rank-based capacity ranking (top 20%) | Stabilizes call volume against shifting base response rates (Notebook 06) |
| D15 | Interpretability | Permutation / SHAP | Permutation importance & TreeSHAP | Provides dual-level global and local auditable explanations (Notebook 07) |
| D16 | Protected Attributes | Keep / Exclude | Audited for responsible use | Exclusion testing demonstrates negligible impact on ranking efficacy (Notebook 07) |


---
## 4 · Reproducibility and Environment


In [3]:
print("Python        :", platform.python_version())
print("pandas / numpy:", pd.__version__, "/", np.__version__)
print("scikit-learn  :", sklearn.__version__)
print("Random seed   :", SEED)
print("Dataset       : UCI Bank Marketing (id 222), bank-additional-full.csv")
print("Shape         :", data["raw"].shape, "| after de-duplication:", data["df"].shape)
print("Fingerprint   :", fingerprint(data["raw"]))
print("Business config: budget =", CALL_BUDGET, "| cost =", COST_PER_CALL, "| value =", VALUE_PER_SUBSCRIPTION)
print("Macro excluded:", EXCLUDE_MACRO, "| adopted feature families:", ADOPTED_FAMILIES or "none")

Python        : 3.14.2
pandas / numpy: 3.0.1 / 2.4.4
scikit-learn  : 1.8.0
Random seed   : 42
Dataset       : UCI Bank Marketing (id 222), bank-additional-full.csv
Shape         : (41188, 21) | after de-duplication: (41176, 21)


Fingerprint   : 00de77bb3ece5d5fdfdd09ec12392284dac89d6d877ec1659092eb5b3522abe1
Business config: budget = 0.2 | cost = 1.0 | value = 10.0
Macro excluded: True | adopted feature families: none


### Academic Declaration
All data preprocessing pipelines, model architectures, validation protocols, and analytical interpretations in this project were implemented, validated, and documented by the project group members in accordance with academic integrity guidelines.

### References
- Moro, S., Cortez, P., & Rita, P. (2014). A data-driven approach to predict the success of bank telemarketing. *Decision Support Systems*, 62, 22-31.
- Batista, G. E., Prati, R. C., & Monard, M. C. (2004). A study of the behavior of several methods for balancing machine learning training data. *ACM SIGKDD Explorations*, 6(1), 20-29.
- Ke, G., et al. (2017). LightGBM: A highly efficient gradient boosting decision tree. *Advances in Neural Information Processing Systems* 30.
- Lundberg, S. M., et al. (2020). From local explanations to global understanding with explainable AI for trees. *Nature Machine Intelligence*, 2, 56-67.
